In [1]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, kendalltau
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt



In [ ]:
# 1 Carga de datos
file_path = "sandiego.csv"
df = pd.read_csv(file_path)

In [ ]:
# 2 Seleccion de variables de interés
variables = [
    'host_response_rate',
    'host_acceptance_rate',
    'host_total_listings_count',
    'accommodates',
    'reviews_per_month',
    'price'
]
df_selected = df[variables].copy

df_selected = df[[
    'host_response_rate',
    'host_acceptance_rate',
    'host_total_listings_count',
    'accommodates',
    'reviews_per_month',
    'price'
]].copy()

In [ ]:
# 3 Limpieza de datos
## Conversion de porcentajes y precios
df_selected['host_response_rate'] = df_selected['host_response_rate'].str.rstrip('%').astype(float)
df_selected['host_acceptance_rate'] = df_selected['host_acceptance_rate'].str.rstrip('%').astype(float)
df_selected['price'] = df_selected['price'].replace('[\$,]', '', regex=True).astype(float)

<>:7: SyntaxWarning: invalid escape sequence '\$'
<>:7: SyntaxWarning: invalid escape sequence '\$'
C:\Users\Maugo\AppData\Local\Temp\ipykernel_25008\1380252658.py:7: SyntaxWarning: invalid escape sequence '\$'
  df_selected['price'] = df_selected['price'].replace('[\$,]', '', regex=True).astype(float)


In [ ]:
# 4 Preprocesamiento: Imputación de nulos con la mediana
df_clean = df_selected.fillna(df_selected.median(numeric_only=True))


In [ ]:
# 5 Preprocesamiento: Eliminación de outliers con IQR
def eliminar_outliers(df, columnas):
    for col in columnas:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    return df

df_sin_outliers = eliminar_outliers(df_clean, df_clean.columns)


In [ ]:
# 6 Análisis de correlación no lineal (Spearman y Kendall)
correlaciones = []
for x in variables:
    for y in variables:
        if x != y:
            try:
                spearman_corr, _ = spearmanr(df_sin_outliers[x], df_sin_outliers[y])
                kendall_corr, _ = kendalltau(df_sin_outliers[x], df_sin_outliers[y])
                correlaciones.append({
                    'Variable X': x,
                    'Variable Y': y,
                    'Spearman': round(spearman_corr, 4),
                    'Kendall': round(kendall_corr, 4)
                })
            except:
                correlaciones.append({
                    'Variable X': x,
                    'Variable Y': y,
                    'Spearman': np.nan,
                    'Kendall': np.nan
                })

df_correlaciones = pd.DataFrame(correlaciones)


C:\Users\Maugo\AppData\Local\Temp\ipykernel_25008\3362473560.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(df_sin_outliers[x], df_sin_outliers[y])


In [ ]:
# 7 Regresión no lineal (polinomial grado 2)
regresion_resultados = []
for x_var in variables:
    for y_var in variables:
        if x_var != y_var:
            X = df_sin_outliers[[x_var]].values
            y = df_sin_outliers[y_var].values

            poly = PolynomialFeatures(degree=2)
            X_poly = poly.fit_transform(X)

            model = LinearRegression().fit(X_poly, y)
            y_pred = model.predict(X_poly)

            r2 = r2_score(y, y_pred)

            try:
                spearman_corr, _ = spearmanr(df_sin_outliers[x_var], df_sin_outliers[y_var])
                kendall_corr, _ = kendalltau(df_sin_outliers[x_var], df_sin_outliers[y_var])
            except:
                spearman_corr, kendall_corr = np.nan, np.nan

            regresion_resultados.append({
                'Variable X': x_var,
                'Variable Y': y_var,
                'Modelo': 'Polinomial grado 2',
                'R^2': round(r2, 4),
                'Spearman': round(spearman_corr, 4),
                'Kendall': round(kendall_corr, 4)
            })


C:\Users\Maugo\AppData\Local\Temp\ipykernel_25008\3947542271.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(df_sin_outliers[x_var], df_sin_outliers[y_var])
C:\Users\Maugo\AppData\Local\Temp\ipykernel_25008\3947542271.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(df_sin_outliers[x_var], df_sin_outliers[y_var])
C:\Users\Maugo\AppData\Local\Temp\ipykernel_25008\3947542271.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(df_sin_outliers[x_var], df_sin_outliers[y_var])
C:\Users\Maugo\AppData\Local\Temp\ipykernel_25008\3947542271.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(df_sin_outliers[x_var], df_sin_outliers[y_var])
C:\Users\Maugo\AppData\Local\Temp\ip

In [15]:
# Crear DataFrame de resultados finales
df_regresion = pd.DataFrame(regresion_resultados)
df_regresion

,Variable X,Variable Y,Modelo,R^2,Spearman,Kendall
0,host_response_rate,host_acceptance_rate,Polinomial grado 2,0.0000,NaN,NaN
1,host_response_rate,host_total_listings_count,Polinomial grado 2,0.0000,NaN,NaN
2,host_response_rate,accommodates,Polinomial grado 2,0.0000,NaN,NaN
3,host_response_rate,reviews_per_month,Polinomial grado 2,0.0000,NaN,NaN
4,host_response_rate,price,Polinomial grado 2,0.0000,NaN,NaN
5,host_acceptance_rate,host_response_rate,Polinomial grado 2,1.0000,NaN,NaN
6,host_acceptance_rate,host_total_listings_count,Polinomial grado 2,0.0133,-0.0834,-0.0645
7,host_acceptance_rate,accommodates,Polinomial grado 2,0.0005,0.0341,0.0275
8,host_acceptance_rate,reviews_per_month,Polinomial grado 2,0.0057,-0.0063,-0.0042
9,host_acceptance_rate,price,Polinomial grado 2,0.0002,0.0264,0.0199


In [16]:
# Mostrar resumen
df_regresion.head()

,Variable X,Variable Y,Modelo,R^2,Spearman,Kendall
0,host_response_rate,host_acceptance_rate,Polinomial grado 2,0.0,NaN,NaN
1,host_response_rate,host_total_listings_count,Polinomial grado 2,0.0,NaN,NaN
2,host_response_rate,accommodates,Polinomial grado 2,0.0,NaN,NaN
3,host_response_rate,reviews_per_month,Polinomial grado 2,0.0,NaN,NaN
4,host_response_rate,price,Polinomial grado 2,0.0,NaN,NaN
